<a href="https://colab.research.google.com/github/deepan98raj-dotcom/My_Project/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers scikit-learn pandas numpy spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 75.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import joblib
import numpy as np
import pandas as pd
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

# ==========================================
# STEP 1: Initialize Models & Expanded Data
# ==========================================
print("Loading spaCy and SentenceTransformer models...")
nlp = spacy.load("en_core_web_sm")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Dataset extended to provide sufficient samples per class
data = {
    "text": [
        # Account Management
        "How do I reset my account password?",
        "I need help recovering my multi-factor authentication token.",
        "Unable to log into my user profile.",
        "How can I change my billing email address?",
        "Locked out of my account after multiple password attempts.",
        # Tech Support
        "The server crashed due to out of memory exception.",
        "Database connections are timing out repeatedly.",
        "API endpoint returning 500 internal server error.",
        "High CPU utilization on the main backend server.",
        "System latency increased following the database migration.",
        # Sports
        "Our team won the championship game last night!",
        "What a stunning goal scored in the final minutes of the match!",
        "The quarterback threw a 50 yard touchdown pass.",
        "Who won the basketball playoff finals yesterday?",
        "The tournament was postponed due to heavy rain.",
    ],
    "category": [
        "Account Management",
        "Account Management",
        "Account Management",
        "Account Management",
        "Account Management",
        "Tech Support",
        "Tech Support",
        "Tech Support",
        "Tech Support",
        "Tech Support",
        "Sports",
        "Sports",
        "Sports",
        "Sports",
        "Sports",
    ],
}
df = pd.DataFrame(data)

# ==========================================
# STEP 2: Text Preprocessing Pipeline
# ==========================================
def preprocess_text(text: str) -> str:
    """Cleans text using spaCy by lowercasing, removing stopwords/punctuation, and extracting lemmas."""
    doc = nlp(text.lower())
    clean_tokens = [
        token.lemma_
        for token in doc
        if not token.is_stop and not token.is_punct and token.is_alpha
    ]
    return " ".join(clean_tokens)

print("Preprocessing text dataset...")
df["clean_text"] = df["text"].apply(preprocess_text)

# ==========================================
# STEP 3: Generate Dense Vector Embeddings
# ==========================================
print("Generating vector embeddings...")
embeddings = embedder.encode(df["clean_text"].tolist(), show_progress_bar=False)

# ==========================================
# STEP 4: Stratified Train-Test Split & SVM Training
# ==========================================
# stratify=df['category'] prevents missing classes in the test set
X_train, X_test, y_train, y_test = train_test_split(
    embeddings,
    df["category"],
    test_size=0.20,
    random_state=42,
    stratify=df["category"],
)

classifier = SVC(kernel="rbf", C=1.0, probability=True)
classifier.fit(X_train, y_train)

# ==========================================
# STEP 5: Model Evaluation
# ==========================================
print("\n=== Model Evaluation ===")
predictions = classifier.predict(X_test)
print(classification_report(y_test, predictions, zero_division=0))

accuracy = accuracy_score(y_test, predictions)
f1_macro = f1_score(y_test, predictions, average="macro", zero_division=0)
print(f"Overall Accuracy: {accuracy * 100:.2f}%")
print(f"Macro F1-Score:   {f1_macro:.4f}\n")

# ==========================================
# STEP 6: Save Model Artifact to Disk
# ==========================================
model_filename = "intent_svm_classifier.joblib"
joblib.dump(classifier, model_filename)
print(f"Model successfully saved to '{model_filename}'")

# ==========================================
# STEP 7: Reload & Test Pipeline
# ==========================================
loaded_classifier = joblib.load(model_filename)

def predict_intent(new_text: str) -> dict:
    """End-to-end inference function using spaCy, SentenceTransformer, and loaded SVM model."""
    cleaned = preprocess_text(new_text)
    vector = embedder.encode([cleaned])

    prediction = loaded_classifier.predict(vector)[0]
    probabilities = loaded_classifier.predict_proba(vector)[0]

    confidence_scores = {
        cls: round(float(prob), 4)
        for cls, prob in zip(loaded_classifier.classes_, probabilities)
    }

    return {
        "query": new_text,
        "predicted_category": prediction,
        "confidence": confidence_scores,
    }

# Run sample test queries
test_queries = [
    "I forgot my login password and cannot access my profile",
    "Database connection pool exhausted during peak loads",
    "Which team scored the final point in the tournament?",
]

print("\n=== Real-Time Inference Test ===")
for query in test_queries:
    result = predict_intent(query)
    print(f"Query:      '{result['query']}'")
    print(f"Predicted:  {result['predicted_category']}")
    print(f"Confidence: {result['confidence']}\n")

Loading spaCy and SentenceTransformer models...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Preprocessing text dataset...
Generating vector embeddings...

=== Model Evaluation ===
                    precision    recall  f1-score   support

Account Management       1.00      1.00      1.00         1
            Sports       1.00      1.00      1.00         1
      Tech Support       1.00      1.00      1.00         1

          accuracy                           1.00         3
         macro avg       1.00      1.00      1.00         3
      weighted avg       1.00      1.00      1.00         3

Overall Accuracy: 100.00%
Macro F1-Score:   1.0000

Model successfully saved to 'intent_svm_classifier.joblib'

=== Real-Time Inference Test ===
Query:      'I forgot my login password and cannot access my profile'
Predicted:  Account Management
Confidence: {'Account Management': 0.9383, 'Sports': 0.0384, 'Tech Support': 0.0234}

Query:      'Database connection pool exhausted during peak loads'
Predicted:  Tech Support
Confidence: {'Account Management': 0.0296, 'Sports': 0.0513, 'Tec